In [50]:
from pathlib import Path
import subprocess
from itertools import combinations

from Bio import SeqIO, SeqRecord, SeqFeature
import yaml
import pandas as pd
from pandas.errors import EmptyDataError

import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

In [31]:
data = Path().resolve() / "synteny/cfr_rivm_mrsa/"

In [32]:
gbffs = []
for file in data.glob('*.gbff'):
    with open(file) as f:
        gbffs.append(SeqIO.read(f, 'genbank'))

In [33]:
rotated = []
for record in gbffs:
    gene_locations = {gene : feature for feature in record.features for gene in feature.qualifiers.get("gene", []) if gene and feature.type == "CDS"}
    goi = gene_locations["cfr"]
    if goi.location.strand == -1:
        record = record.reverse_complement(id=record.id, name=True, description=True, features=True, annotations=True, letter_annotations=True, dbxrefs=True)
        gene_locations = {gene : feature for feature in record.features for gene in feature.qualifiers.get("gene", []) if gene and feature.type == "CDS"}
        goi = gene_locations["cfr"]
    start = goi.location.start
    record = record[start:] + record[:start]
    rotated.append(record)

In [34]:
path = Path(data, 'rotated')
path.mkdir(exist_ok=True)
for record in rotated:
    with open(path / f"{record.id}.gbff", 'w') as handle:
        SeqIO.write(record, handle=handle, format='genbank')

In [35]:
for a, b in combinations(rotated, 2):
    out = path.parent / 'blast'
    out.mkdir(exist_ok=True)

    file_a = out / f"{a.id}.fasta"
    file_b = out / f"{b.id}.fasta"

    with open(file_a, "w") as handle:
        SeqIO.write(a, handle, "fasta")
    
    with open(file_b, "w") as handle:
        SeqIO.write(b, handle, "fasta")

    cmd = f"""
    source activate blast \
        && bsub \
            -o {out / 'blast.out'} \
            -e {out / 'blast.err'} \
                "blastn \
                    -task blastn \
                    -query {file_a}\
                    -subject {file_b}\
                    -evalue 1e-5 \
                    -perc_identity 90 \
                    -out {out / f'{a.id}_{b.id}.csv'} \
                    -outfmt '20 qseqid qlen qstart qend qstrand sseqid slen sstart send sstrand'
                "
    """
    subprocess.run(cmd, shell=True)

Job <37358301> is submitted to queue <bio>.
Job <37358302> is submitted to queue <bio>.
Job <37358303> is submitted to queue <bio>.
Job <37358304> is submitted to queue <bio>.
Job <37358305> is submitted to queue <bio>.
Job <37358306> is submitted to queue <bio>.
Job <37358307> is submitted to queue <bio>.
Job <37358308> is submitted to queue <bio>.
Job <37358309> is submitted to queue <bio>.
Job <37358310> is submitted to queue <bio>.
Job <37358311> is submitted to queue <bio>.
Job <37358312> is submitted to queue <bio>.
Job <37358313> is submitted to queue <bio>.
Job <37358314> is submitted to queue <bio>.
Job <37358315> is submitted to queue <bio>.
Job <37358316> is submitted to queue <bio>.
Job <37358317> is submitted to queue <bio>.
Job <37358318> is submitted to queue <bio>.
Job <37358319> is submitted to queue <bio>.
Job <37358320> is submitted to queue <bio>.
Job <37358321> is submitted to queue <bio>.
Job <37358322> is submitted to queue <bio>.
Job <37358323> is submitted to q

In [37]:
for file in out.glob("*.fasta"):
    mob_out = out.parent / 'mobtyper'
    mob_out.mkdir(exist_ok=True)
    cmd = f"""
    source activate mob-suite \
        && bsub \
            -o {mob_out / 'mobtyper.out'} \
            -e {mob_out / 'mobtyper.err'} \
                "mob_typer \
                    --infile {file} \
                    --out_file {mob_out / f'{file.stem}.mobtyper'} \
                    --biomarker_report_file {mob_out / f'{file.stem}.biomarkers'} \
                "
    """
    subprocess.run(cmd, shell=True)

Job <37358359> is submitted to queue <bio>.
Job <37358360> is submitted to queue <bio>.
Job <37358361> is submitted to queue <bio>.
Job <37358362> is submitted to queue <bio>.
Job <37358363> is submitted to queue <bio>.
Job <37358364> is submitted to queue <bio>.
Job <37358365> is submitted to queue <bio>.
Job <37358366> is submitted to queue <bio>.
Job <37358367> is submitted to queue <bio>.
Job <37358368> is submitted to queue <bio>.


In [38]:
biomarkers = []
for file in mob_out.glob("*.biomarkers"):
    try:
        biomarkers.append(pd.read_table(file, sep = "\t").assign(seq_id=file.stem))
    except EmptyDataError as e:
        print(e, file.stem)
        continue

biomarkers = pd.concat(biomarkers, ignore_index=True)
biomarkers = biomarkers.rename(columns = {
    'sstart' : 'start',
    'send' : 'end',
    'sstrand' : 'strand',
    'biomarker' : 'gene_type'
})
biomarkers = biomarkers[["seq_id", "start", "end", "strand", "gene_type"]]
biomarkers['type'] = 'CDS'
biomarkers["strand"] = biomarkers["strand"].map({"plus" : "+", "minus" : "-"})
biomarkers.to_csv(path.parent / "mob_markers.csv", index=False)

No columns to parse from file pRIVM_M085090_1
No columns to parse from file pRIVM_M047916_1


In [39]:
blast = pd.concat([pd.read_csv(file) for file in out.glob("*.csv")], ignore_index=True)
blast = blast.rename(columns={
    'qseqid' : 'seq_id',
    'qlen' : 'length',
    'qstart' : 'start',
    'qend' : 'end',
    'sseqid' : 'seq_id2',
    'slen' : 'length2',
    'sstart' : 'start2',
    'send' : 'end2',
    'sstrand' : 'strand2'
})
blast["strand"] = "+"
blast["strand2"] = blast["strand2"].map({"plus" : "+", "minus" : "-"})
blast.to_csv(path.parent / "blast_homology.csv", index=False)

In [41]:
cmd = f"""
source activate mash \
    && bsub \
        -o {data / 'mash.out'} \
        -e {data / 'mash.err'} \
            "mash \
                triangle \
                -k 21 \
                -s 10000 \
                {' '.join(str(file) for file in data.glob('blast/*.fasta'))} \
                > {data / 'mash_dist.tsv'}
            "
"""
subprocess.run(cmd, shell=True)

Job <37358370> is submitted to queue <bio>.


CompletedProcess(args='\nsource activate mash     && bsub         -o /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/mash.out         -e /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/mash.err             "mash                 triangle                 -k 21                 -s 10000                 /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/blast/pRIVM_M044329_1.fasta /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/blast/pRIVM_M041248_1.fasta /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/blast/pRIVM_M045402_1.fasta /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/blast/pRIVM_M085090_1.fasta /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/blast/pRIVM_M083782_1.fasta /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/blast/pRIVM_M096426_1.fasta /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/synteny/cfr_rivm_mrsa/blast/pRIVM_M043455_2.fasta /mnt/s

In [42]:
with open(data / 'mash_dist.tsv') as file:
    mash = [[value.strip() for value in row.split('\t')] for row in file.readlines()][1:]

In [44]:
names = [Path(row[0]).stem for row in mash]
distances = [value for row in mash for value in row[1:]]

In [48]:
matrix = np.zeros((len(mash), len(mash)))
matrix[np.tril_indices_from(matrix, k=-1)] = distances
matrix = matrix + matrix.T
m = squareform(matrix)
z = linkage(m, 'single')

In [55]:
with open(data / "synteny_order.txt", "w") as file:
    file.write("\n".join([names[i] for i in leaves_list(z)]))